# Task 1 : Imaging factors that may distort T2 relaxometry

Explore acquisition artefacts, monotonicity, SNR, partial volume, and mono-exponentiality across the preterm cohort.

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore', category=RuntimeWarning)

import numpy as np
import pandas as pd
from utils import load_preterm_cohort, load_adult_case, mean_decay_curve
from quality import (check_monotonicity, estimate_sigma_bg, subject_logfit_score,
                     boundary_fractions, cohort_qa_table)
from plotting import (plot_echo_gallery, plot_seg_overlay_slices,
                      plot_soft_seg_channels, plot_tissue_decay_curves,
                      plot_voxel_heterogeneity, plot_monotonicity_map,
                      plot_snr_curves, plot_logfit_residuals, plot_qa_deep_dive)

## 1.1 : Load the analysis cohort

In [ ]:
preterm, cohort = load_preterm_cohort('../data')
adults = [load_adult_case(i) for i in range(1, 7)]

print(f"Preterm analysis cohort: n = {len(preterm)} "
      f"(EP = {sum(1 for d in preterm if d['EP'])}, "
      f"FT = {sum(1 for d in preterm if not d['EP'])})")
print(f"TE schedule: {preterm[0]['TE']}")
print(f"Adult controls: {len(adults)} subjects loaded")

## 1.2 : How images change with echo time

In [ ]:
plot_echo_gallery(preterm[:3])

## 1.3 : Tissue segmentation overlay

In [ ]:
plot_seg_overlay_slices(preterm[:2])
plot_soft_seg_channels(preterm[0])

## 1.4 : Mean decay curves by tissue

In [ ]:
plot_tissue_decay_curves(preterm[:3])

## 1.5 : Voxel-level heterogeneity within WM

In [ ]:
plot_voxel_heterogeneity(preterm[0], seg_idx=3, n_voxels=200)

## 1.6 : Monotonicity test

In [ ]:
df_mono, df_logfit, qa = cohort_qa_table(preterm)
print("Monotonicity summary:")
print(df_mono.describe().round(2).to_string())

# Spatial map of violations
_, _, viol_map = check_monotonicity(preterm[0]['reg'], preterm[0]['mask'])
plot_monotonicity_map(preterm[0], viol_map)

## 1.7 : Mono-exponentiality test via log-linear residuals

In [ ]:
plot_logfit_residuals(preterm[0])

print("\nConsolidated QA (worst 10% on either metric):")
print(f"  Flagged: {int(qa['flag'].sum())} / {len(qa)}")

## 1.8 : SNR and the Rician noise floor

In [ ]:
plot_snr_curves(preterm)

## 1.9 : Partial volume at tissue boundaries

In [ ]:
rows = []
for ds in preterm:
    bf = boundary_fractions(ds)
    rows.append({t: round(bf[k]*100, 1) for k, t in [(1,'CSF'), (2,'GM'), (3,'WM')]})
print("Boundary fraction (%) per tissue:")
print(pd.DataFrame(rows).agg(['median', 'mean', 'std']).round(1).to_string())

## 1.10 : Deep dive on flagged subjects and exclusion list

In [ ]:
id_to_ds = {ds['id']: ds for ds in preterm}
worst_mono = qa.loc[qa['worst_10pct_mono']].sort_values('monotonicity_viol_pct', ascending=False)
plot_qa_deep_dive('Worst 10% on monotonicity', worst_mono, id_to_ds)

# Save cohort files for downstream notebooks
preterm_ids_df = pd.DataFrame({'id': [ds['id'] for ds in preterm],
                                'EP': [ds['EP'] for ds in preterm]})
preterm_ids_df.to_csv('../data/preterm_ids.csv', index=False)

exclude_ids = ['17065', '48996', '96179']
pd.DataFrame({'id': exclude_ids}).to_csv('../data/preterm_exclusions.csv', index=False)
print(f"Saved preterm_ids.csv ({len(preterm_ids_df)} subjects) and exclusions ({len(exclude_ids)})")